In [309]:
# Loading the data
import json
import pandas as pd
from pathlib import Path
import sys
import matplotlib.pyplot as plt
from sklearn.covariance import MinCovDet
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Load data/valencia_cf_1/matches.json
matches_path = project_root / "tests" / "fixtures" / "testing_data" / "data" / "valencia_cf_1" / "matches.json"
raw_matches = json.load(open(matches_path))
print(f"Loaded {len(raw_matches)} matches from {matches_path}")

Loaded 93 matches from C:\Users\kazik\Projects\Gaffers-Clipboard\tests\fixtures\testing_data\data\valencia_cf_1\matches.json


In [310]:
normalized_df = pd.json_normalize(
    raw_matches,
    record_path="player_performances",
    meta=[
        "id", ["data", "half_length"], 
        ["data", "home_team_name"], 
        ["data", "away_team_name"], 
        ["data", "home_stats", "possession"], 
        ["data", "away_stats", "possession"],
        ["data", "home_stats", "xg"],
        ["data", "away_stats", "xg"],
        ["data", "home_stats", "passes"],
        ["data", "away_stats", "passes"]
    ]
)
# Create 4 new columns: team_possession, team_xg, opponent_xg, and team_passes, and set all the values to NaN
normalized_df["team_possession"] = float("nan")
normalized_df["team_xg"] = float("nan")
normalized_df["opponent_xg"] = float("nan")
normalized_df["team_passes"] = float("nan")

for index, performance in normalized_df.iterrows():
    # Check if the performance is for the home team or away team
    if performance["data.home_team_name"] == "Valencia CF":
        normalized_df.at[index, "team_possession"] = performance["data.home_stats.possession"]
        normalized_df.at[index, "team_xg"] = performance["data.home_stats.xg"]
        normalized_df.at[index, "opponent_xg"] = performance["data.away_stats.xg"]
        normalized_df.at[index, "team_passes"] = performance["data.home_stats.passes"]
    else:
        normalized_df.at[index, "team_possession"] = performance["data.away_stats.possession"]
        normalized_df.at[index, "team_xg"] = performance["data.away_stats.xg"]
        normalized_df.at[index, "opponent_xg"] = performance["data.home_stats.xg"]
        normalized_df.at[index, "team_passes"] = performance["data.away_stats.passes"]

# Remove the home and away names, possession, xg, and passes columns
normalized_df = normalized_df.drop(columns=["data.home_team_name", "data.away_team_name", "data.home_stats.possession", "data.away_stats.possession", "data.home_stats.xg", "data.away_stats.xg", "data.home_stats.passes", "data.away_stats.passes"])

normalized_df = normalized_df.rename(columns={"id": "match_id"})

normalized_df = normalized_df[normalized_df["performance_type"] != "GK"]

normalized_df = normalized_df.dropna(axis=1, how="all")

# Now we want to filter for players with only "CM" in "positions_played"
cm_players_df = normalized_df[normalized_df["positions_played"].apply(lambda x: x == ["CM"])]

# Now remove columns that are all NaN
cm_players_df = cm_players_df.dropna(axis=1, how="all")

# Remove "performance_type" and "positions_played" columns
cm_players_df = cm_players_df.drop(columns=["performance_type", "positions_played"])

# Move "match_id" to the front, and rename the id column to "performance_id"
cols = cm_players_df.columns.tolist()
cols.insert(0, cols.pop(cols.index("match_id")))
cm_players_df = cm_players_df[cols]
cm_players_df.head()

# Rename "data.half_length" to "half_length"
cm_players_df = cm_players_df.rename(columns={"data.half_length": "half_length"})

# Output number of rows and columns, and the first few rows of the resulting dataframe
print(f"Number of rows: {cm_players_df.shape[0]}")
print(f"Number of columns: {cm_players_df.shape[1]}")
print("Columns:")
print(cm_players_df.columns.tolist())
print("\n")

# Output the max and min of every column, to get a sense of the range of values
for col in cm_players_df.columns:
    if col in ["match_id", "player_id"]:
        continue
    if cm_players_df[col].dtype in ["int64", "float64"]:
        print(f"{col}: min={cm_players_df[col].min()}, max={cm_players_df[col].max()}")

Number of rows: 251
Number of columns: 24
Columns:
['match_id', 'player_id', 'goals', 'assists', 'shots', 'shot_accuracy', 'passes', 'pass_accuracy', 'dribbles', 'dribble_success_rate', 'tackles', 'tackle_success_rate', 'offsides', 'fouls_committed', 'possession_won', 'possession_lost', 'minutes_played', 'distance_covered', 'distance_sprinted', 'half_length', 'team_possession', 'team_xg', 'opponent_xg', 'team_passes']


goals: min=0.0, max=8.0
assists: min=0.0, max=8.0
shots: min=0.0, max=12.0
shot_accuracy: min=0.0, max=100.0
passes: min=0.0, max=55.0
pass_accuracy: min=0.0, max=100.0
dribbles: min=0.0, max=54.0
dribble_success_rate: min=0.0, max=100.0
tackles: min=0.0, max=12.0
tackle_success_rate: min=0.0, max=100.0
offsides: min=0.0, max=8.0
fouls_committed: min=0.0, max=8.0
possession_won: min=0.0, max=8.0
possession_lost: min=0.0, max=14.0
minutes_played: min=2.0, max=94.0
distance_covered: min=0.3, max=14.1
distance_sprinted: min=0.1, max=8.0
team_possession: min=39.0, max=71.0


In [311]:
# 1. Select every column EXCEPT your identifiers
cols_to_convert = cm_players_df.columns.drop(['match_id', 'player_id'])

# 2. Force them all to numeric in one swoop
cm_players_df[cols_to_convert] = cm_players_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# Optional: If any weird JSON text like "N/A" got turned into a NaN, 
# you can safely fill them with 0s to keep the math from breaking later.
cm_players_df[cols_to_convert] = cm_players_df[cols_to_convert].fillna(0)

In [312]:
# Step 1: Removing players with less than 10 minutes played
cm_players_df = cm_players_df[cm_players_df["minutes_played"] >= 10]

In [313]:
# Step 1: Volume masking
# For each percentage stat, their associated volume attempted stat must reach a certain threshold for the percentage 
# stat to be considered valid. For example, if a player attempted 0 passes, their pass accuracy should not be considered valid,
# and should be set to the mean.
volume_perc_pairs = [
    ("passes", "pass_accuracy"),
    ("shots", "shot_accuracy"),
    ("dribbles", "dribble_success_rate"),
    ("tackles", "tackle_success_rate")
]
volume_masks = {
    "passes": 5,
    "shots": 2,
    "dribbles": 3,
    "tackles": 2
}

for volume_col, perc_col in volume_perc_pairs:
    # apply the mask
    valid = cm_players_df[cm_players_df[volume_col] >= volume_masks[volume_col]]
    mean_value = valid[perc_col].mean()
    cm_players_df[perc_col] = cm_players_df.apply(
        lambda r: r[perc_col] if r[volume_col] >= volume_masks[volume_col] else mean_value,
        axis=1
    )

In [314]:
# ---------------------------------------------------------
# Step 1: Standardize Absolute Volume (The Half-Length Fix)
# ---------------------------------------------------------
cum_columns = ["goals", "assists", "shots", "passes", "dribbles", "tackles", 
               "possession_won", "possession_lost", "fouls_committed", "offsides", 
               "distance_covered", "distance_sprinted"]

h_base = 10.0

for col in cum_columns:
    # We overwrite the raw stat to represent a standard 10-minute half game.
    # This ensures 5-min half games don't corrupt our dataset averages later.
    cm_players_df[col] = cm_players_df[col] * (h_base / cm_players_df["half_length"])


# ---------------------------------------------------------
# Step 2: Set Bayesian Smoothing Parameters
# ---------------------------------------------------------
dummy_minutes = 30.0  # Shrinkage anchor (requires ~a half of football to break away from mean)


# ---------------------------------------------------------
# Step 3: Apply Smoothed Per-90 Scaling
# ---------------------------------------------------------
# A. Rare Stats (Assume 0 for dummy minutes)
rare_cols = ["goals", "assists", "shots"]
for col in rare_cols:
    cm_players_df[f"{col}_p90"] = (cm_players_df[col] / (cm_players_df["minutes_played"] + dummy_minutes)) * 90.0


# B. Volume Stats (Assume dataset average for dummy minutes)
volume_cols = ["passes", "dribbles", "tackles", "possession_won", "possession_lost", 
               "fouls_committed", "offsides", "distance_covered", "distance_sprinted"]

for col in volume_cols:
    # 1. Calculate the true, half-length-adjusted dataset average Per 90
    league_avg_p90 = (cm_players_df[col].sum() / cm_players_df["minutes_played"].sum()) * 90.0
    
    # 2. Calculate the dummy stat allocation for 45 minutes
    dummy_stat = league_avg_p90 * (dummy_minutes / 90.0)
    
    # 3. Apply the final smoothed formula
    cm_players_df[f"{col}_p90"] = ((cm_players_df[col] + dummy_stat) / (cm_players_df["minutes_played"] + dummy_minutes)) * 90.0

# Print to verify
print(f"Passes mean after smoothing: {cm_players_df['passes_p90'].mean():.2f}, std: {cm_players_df['passes_p90'].std():.2f}")

Passes mean after smoothing: 35.07, std: 6.62


In [315]:
cm_players_df["xT_bonus_p90"] = 0.25 * (cm_players_df["distance_sprinted_p90"] / cm_players_df["distance_covered_p90"]) * np.log(cm_players_df["pass_accuracy"] * cm_players_df["passes_p90"] + 1)

In [316]:
# Columns to log for PCA only; do NOT modify cm_players_df in-place
cols_to_log = ["goals_p90", "assists_p90", "shots_p90", "offsides_p90", "fouls_committed_p90", "possession_won_p90", "possession_lost_p90"]

**Adjusting for match supremacy**<br>
The JSON metadata dictates the match is played on Ultimate difficulty. Furthermore, player ratings must accurately reflect team dominance to align with the variance models mapped previously, where exactly 26% of rating variance was proven to be team-dependent. We calculate the Match Supremacy Scalar ($\Delta_{xG}$) utilizing the aggregate home_stats and away_stats extracted from the JSON:
$$\Delta_{xG} = \frac{xG_{team} + 1}{xG_{opponent} + 1}$$
If $\Delta_{xG} < 1$ (indicating the team was mathematically dominated by the opponent) but a specific player still achieved a high raw aggregate $S_i$, their performance is statistically far more valuable than achieving the exact same output within a heavily dominant, high-possession team. To account for this, the final dynamic score adjusts the slope of the rating based on team supremacy:
$$R_{final} = R_{raw} + \gamma \ln(\Delta_{xG})$$
Where $\gamma$ is a conservative tuning parameter set to 0.25, ensuring the adjustment shifts the rating by no more than $\pm 0.25$ points.

This is calculated now, and then added on to the final rating after the PCA weights have been applied, to ensure that players on weaker teams who perform well are not undervalued, and players on stronger teams who perform well are not overvalued, creating a more balanced and accurate match rating system that accounts for the context of the player's performance.

In [317]:
# Match supremacy scalar
gamma = 0.25
delta_xg = (cm_players_df['team_xg'] + 1) / (cm_players_df['opponent_xg'] + 1)
match_supremacy_scalar = gamma * np.log(delta_xg)

In [318]:
# Now we import weights from root/workshop/cm_weights.json
# Which is in the format {"goals_p90": 0.3, "assists_p90": 0.2, ...}
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
weights_path = project_root / "workshop" / "cm_weights.json"
with open(weights_path, "r") as f:
    weights_dict = json.load(f)
final_weights = np.array([weights_dict[col] for col in col_names])

In [319]:
# Now we import the means and stds from root/workshop/cm_means_stds.json
# Which is in the format {"goals_p90": {"mean": 0.1, "std": 0.2}, "assists_p90": {"mean": 0.05, "std": 0.1}, ...}
means_stds_path = project_root / "workshop" / "cm_means_stds.json"
with open(means_stds_path, "r") as f:
    means_stds_dict = json.load(f)

### Creating the final match rating algorithm
Now that we have the weights for each stat, we can create the final match rating algorithm.<br>
The raw aggregate score $S_i$ for the $i$-th player is the mathematical dot product of their standardized feature vector $Z_i$ and the transposed positional weight vector $\lambda_{pos}$:
$$S_i = Z_i \cdot \lambda_{pos}^T$$
By executing this via NumPy's @ operator or np.dot(), the entire team's aggregate standard deviations are calculated instantaneously.

In [320]:
# Compute final z-scores
negative_stats = ["fouls_committed_p90", "possession_lost_p90", "offsides_p90"]
for col in col_names:
    mean = means_stds_dict[col]["mean"]
    std = means_stds_dict[col]["std"]
    if std == 0:
        cm_players_df[f"{col}_z"] = 0
    elif col in negative_stats:
        cm_players_df[f"{col}_z"] = (mean - cm_players_df[col]) / std
    else:
        cm_players_df[f"{col}_z"] = (cm_players_df[col] - mean) / std

In [321]:
# Prevent missing out on luxury/situational stats from destroying a rating
cols_to_floor = ['assists_p90_z', 'goals_p90_z']
for col in cols_to_floor:
    # Caps the negative penalty at -0.5 standard deviations
    cm_players_df[col] = np.clip(cm_players_df[col], a_min=-0.5, a_max=None)

# Prevent bad luck/poor efficiency from generating black-hole Z-scores
cols_to_floor = [
    'tackle_success_rate_z', 'shot_accuracy_z', 'pass_accuracy_z'
]

# Estimate score before any protective clipping
z_score_cols = [f"{col}_z" for col in col_names]
pre_cap_raw_score = np.dot(cm_players_df[z_score_cols].values, final_weights)

# With your current sigmoid calibration, raw_score < 0 maps to rating < 6.0
low_rating_mask = pre_cap_raw_score < 0.0

# Luxury/situational stats: cap only for low-rated players
for col in ["assists_p90_z", "goals_p90_z"]:
    cm_players_df.loc[low_rating_mask, col] = np.clip(
        cm_players_df.loc[low_rating_mask, col], a_min=-0.5, a_max=None
    )

# Efficiency stats: cap only for low-rated players
for col in ["tackle_success_rate_z", "shot_accuracy_z", "pass_accuracy_z"]:
    cm_players_df.loc[low_rating_mask, col] = np.clip(
        cm_players_df.loc[low_rating_mask, col], a_min=-0.75, a_max=None
    )

cm_players_df["goals_p90_z"] = np.clip(cm_players_df["goals_p90_z"], a_min=None, a_max=1.5)

In [322]:
z_score_cols = [f"{col}_z" for col in col_names]
z_matrix = cm_players_df[z_score_cols].values
cm_players_df['raw_score'] = np.dot(z_matrix, final_weights)

In [323]:
# Add bonuses for goals and assists only when the z-score is positive
cm_players_df['raw_score'] += np.where(cm_players_df['goals_p90_z'] > 0, cm_players_df['goals_p90_z'] * 0.6, 0)
cm_players_df['raw_score'] += np.where(cm_players_df['assists_p90_z'] > 0, cm_players_df['assists_p90_z'] * 0.5, 0)

To map the unbounded continuous variable $S_i \in (-\infty, \infty)$ to the recognized $0.0 - 10.0$ football rating continuum, we apply the inverse logistic function discussed previously.<br>
As identified in the commercial analysis, WhoScored utilizes a base of 6.0 while Sofascore utilizes 6.5. For this custom algorithm, an exact median baseline of $6.0$ is targeted for an entirely average performance (where the sum of standard deviations $S_i = 0$).
$$R_{raw} = 10 \times \left( \frac{1}{1 + e^{-k(S_i - S_0)}} \right)$$
We solve for the translation parameter $S_0$ such that $R_{raw} = 6.0$ when $S_i = 0$:
$$6.0 = \frac{10}{1 + e^{k \cdot S_0}}$$
$$1 + e^{k \cdot S_0} = \frac{10}{6.0} = 1.666$$
$$e^{k \cdot S_0} = 0.666$$
$$k \cdot S_0 = \ln(0.666) \approx -0.405$$
Setting the steepness parameter $k = 0.85$ provides an optimal curve. This specific slope allows for $3\sigma$ events (such as a multi-goal performance) to mathematically approach scores of 9.5 without hitting a rigid asymptote of 10.0 too early in the distribution. Solving for $S_0$ with $k=0.85$ yields $S_0 \approx -0.477$.<br><br>
Thus, the final formula for the raw rating before the match supremacy adjustment is:
$$R_{raw} = 10 \times \left( \frac{1}{1 + e^{-0.85(S_i + 0.477)}} \right)$$

In [324]:
k = 0.85
s_0 = np.log(2/3) / k
# Create a new column, applying the sigmoid transformation to the raw score
cm_players_df['raw_rating'] = 10 * (1 / (1 + np.exp(-k * (cm_players_df['raw_score'] - s_0))))

In [325]:
# Now create a final rating column that subtracts the match supremacy scalar from the raw rating
cm_players_df['final_rating'] = cm_players_df['raw_rating'] - match_supremacy_scalar

In [326]:
# Now a final clamp, to ensure that the final rating is between 0 and 10
cm_players_df['final_rating'] = cm_players_df['final_rating'].apply(lambda x: max(0, min(10, x)))

In [346]:
# Find the row number of the performance of player id 31, in match id 63
cm_players_df.iloc[211]

match_id                          82
player_id                         24
goals                            0.0
assists                          0.0
shots                            3.0
shot_accuracy                   67.0
passes                          54.0
pass_accuracy                   91.0
dribbles                        54.0
dribble_success_rate            98.0
tackles                         10.0
tackle_success_rate             40.0
offsides                         0.0
fouls_committed                  0.0
possession_won                   8.0
possession_lost                  6.0
minutes_played                  91.0
distance_covered                13.7
distance_sprinted                6.3
half_length                       10
team_possession                 66.0
team_xg                          5.3
opponent_xg                      1.4
team_passes                    271.0
goals_p90                        0.0
assists_p90                      0.0
shots_p90                   2.231405
p